# Session 3 · Working with Real Data II + Meet scikit-learn

**Machine Learning Foundations · Sanketana School of Code**

Last session you cleaned a real dataset. But clean is not the same as *ready*. A model is maths, and maths only eats numbers — so a column of words and columns on wildly different scales still stand in the way. Today we clear both, then meet **scikit-learn**, the library that builds every model from here on.

By the end of this notebook you will be able to:

- one-hot encode a text column, and say why numbering categories `0`/`1`/`2` lies
- explain why features on different scales can be a problem, and apply a scaler
- use the scikit-learn **`fit` / `predict` / `score`** trio on a black-box model
- split data into train and test, and explain *why we hide test data*

**How this notebook works:** run every cell top to bottom with your coach. ✏️ cells are yours. Every cell already runs before you fill the gaps.

## Warm-up · Last session's homework

Your coach will walk through Session 2's homework with you (about 10 minutes).

✏️ One thing to carry into today: that phone dataset still had a **text** column (`brand`). A model can't do arithmetic on the word *iPhone*. So what *do* we do with a column of words? That is the first thing we solve today.

## Part 1 · Reload and re-clean (a quick Session 2 recap)

We pick up `housing.csv` exactly where we left it. First, the two-line clean from last session: fill the missing values with each column's median.

In [ ]:
import pandas as pd

homes = pd.read_csv("../../../datasets/secondary/housing.csv")

# Session 2 recap: fill the two columns that had missing values.
homes["age_years"] = homes["age_years"].fillna(homes["age_years"].median())
homes["distance_to_center_km"] = homes["distance_to_center_km"].fillna(homes["distance_to_center_km"].median())

print("missing values now:", homes.isna().sum().sum())
homes.head()

## Part 2 · Turn words into numbers (without lying)

`neighborhood_type` is text: `city_center`, `suburb`, `outskirts`. A model can't multiply "suburb."

The tempting shortcut — city_center = 0, suburb = 1, outskirts = 2 — is a **trap**. Those numbers invent an order and spacing the data never had (is outskirts really "twice" a suburb?).

The honest fix is **one-hot encoding**: one yes/no column per category, no fake order.

In [ ]:
print(homes["neighborhood_type"].value_counts())

In [ ]:
# ✏️ TODO: one-hot encode neighborhood_type. (This line already works — read it.)
# dtype=int makes the new columns read as 0/1 instead of True/False.
homes_encoded = pd.get_dummies(homes, columns=["neighborhood_type"], dtype=int)

homes_encoded.head()

### ✏️ Why one-hot, not 0/1/2?

1. List the new columns one-hot encoding created. What does a `1` in `neighborhood_type_suburb` mean for that row?
2. In one sentence: what false claim would numbering them `0`/`1`/`2` have made?

*Your answers:*

1.
2.

## Part 3 · Put features on the same footing (scaling)

Look at the spread of two columns. `area_sqft` runs into the thousands; `bedrooms` runs 1–5. Some models treat the big-numbered column as if it matters more — just because its numbers are bigger.

**Scaling** rescales every feature to a comparable range. Watch the before and after.

In [ ]:
numeric_features = ["area_sqft", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
print("BEFORE scaling — averages are on totally different scales:")
print(homes_encoded[numeric_features].mean().round(1))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(homes_encoded[numeric_features])

# Re-wrap as a DataFrame just so we can read it.
scaled_df = pd.DataFrame(scaled, columns=numeric_features)
print("AFTER scaling — every feature now centres near 0 with a comparable spread:")
print(scaled_df.mean().round(2))
print()
print("spread (std) of each, after scaling:")
print(scaled_df.std().round(2))

### ✏️ A note on when this matters

We just *saw* scaling work, but here is the honest part: today's black-box model (next part) does fine **without** it. Distance-based models we meet later — like KNN in Module 3 — care a lot.

So we scale features into the model pipeline only when the model needs it. For today, we'll keep things simple and feed the model the **unscaled, one-hot-encoded** data.

✏️ In your own words: why might a model that measures "distance" between rows be misled by an unscaled `area_sqft`?

*Your answer:* 

## Part 4 · Meet scikit-learn — the three moves

Here is the idea that makes the rest of the course easy. **Every** model in scikit-learn is used with the same three moves:

```
model.fit(X, y)        learn the pattern from training data
model.predict(X_new)   apply the pattern to new rows
model.score(X, y)      how well did it do?  (one number)
```

- `X` is the table of **features** (capital — it's a whole table).
- `y` is the single column we want to predict — the **label** (lower-case — one column).

Today's model is `LinearRegression`. Treat it as a **sealed box**: we run the three moves and read the result. *How* it works inside is Module 2.

In [ ]:
# ✏️ TODO: split the data into X (features) and y (the label).
# y is what we predict: price_lakhs. X is everything else.
y = homes_encoded["price_lakhs"]
X = homes_encoded.drop(columns=["price_lakhs"])

print("X (features) shape:", X.shape)
print("y (label) shape:   ", y.shape)
print("feature columns:", list(X.columns))

## Part 5 · Train, then test — why we hide data

Now Session 1 comes back. Remember the "model" that memorized the answer key and scored a perfect 100% — yet was useless on any new song?

If we train a model and score it on the **same** data, we can be fooled exactly the same way. So we **split first**: train on most of the data, and *hide* a slice as a test set the model never sees while learning. Then we score on that hidden slice.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Hide 20% of the rows as a test set the model will not see while training.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("training rows:", X_train.shape[0], " | hidden test rows:", X_test.shape[0])

# The three moves:
model = LinearRegression()
model.fit(X_train, y_train)                       # 1. learn, from TRAINING data only

train_score = model.score(X_train, y_train)       # score on data it has seen
test_score = model.score(X_test, y_test)          # score on data it has NEVER seen

print()
print("score on training data (already seen):", round(train_score, 3))
print("score on test data (never seen):      ", round(test_score, 3))

In [ ]:
# 2. predict: use the model on a few held-out homes and compare to the truth.
predictions = model.predict(X_test)               # apply the pattern to new rows

for predicted, actual in zip(predictions[:5], y_test[:5]):
    print(f"predicted: {predicted:6.1f} lakhs   actual: {actual:6.1f} lakhs")

### ✏️ Read the two scores

The `.score()` of a regression model is a number called **R²**. We won't unpack it today — just read it as "higher is better, 1.0 is perfect." ("Numbers, not vibes" becomes a firm rule in Session 8.)

1. Was the **training** score higher than the **test** score, lower, or about the same?
2. Our memorizing model in Session 1 scored 100% on seen data and was useless on unseen data. How is the train-vs-test gap the *same idea*, made measurable?
3. **Stretch (predict before you re-run):** if you change `test_size` to `0.5` (hide half the data), do you expect the test score to go up, down, or stay roughly the same? Try it, then say what you saw.

*Your answers:*

1.
2.
3.

## What we learned

✏️ Three quick reflections — one line each:

1. The three scikit-learn moves, from memory:
2. Why we one-hot encode instead of numbering categories 0/1/2:
3. Why we score on data the model has never seen:

---

**The path ahead.** Today you ran every move once, with your coach driving: encode → (scale when needed) → split → `fit` / `predict` / `score`. That is almost the whole workflow.

**Next session:** *you* drive. On a clean dataset, start to finish, you'll build your first model end-to-end — and we'll finally give the workflow its name: **data → model → evaluation → insight.**

**Homework:** `homework.ipynb`, 30–45 minutes. It states its success criterion at the top. Revise with `explainer.md`.